# 02 — `<TASK_NAME>` (TEMPLATE)

> **TEMPLATE** — file này được copy bởi `init_project.ps1` từ `colab-venv-bootstrap/templates/`. Sửa thành `02_<task>.ipynb` theo nhiệm vụ cụ thể của project (train / serve / infer / benchmark / ...).

Notebook này chạy mỗi session Colab. Pattern chung:
1. Mount Drive.
2. Load `colab/config.env` + auto-bootstrap venv nếu runtime mới.
3. Activate venv + chạy task chính.

**Trước khi chạy**:
- Đã chạy `00_clone_repo.ipynb` (repo có trên Drive).
- Đã chạy `01_bootstrap_env.ipynb` ít nhất 1 lần (venv tồn tại hoặc snapshot có trên Drive).

**Tham khảo notebook 02 thực tế**:
- `colab/bootstrap/examples/piper-tts-finetuning/02_train.ipynb` — train TTS với piper-train + checkpoint Drive.
- `colab/bootstrap/examples/fastapi-server-cloudflared/02_run_server.ipynb` — serve FastAPI + Cloudflare tunnel.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import subprocess

# ============================================================
# EDIT 2 biến này (match với notebook 00 + 01)
# ============================================================
PROJECT_SLUG = 'my_project'
BASE_ROOT    = '/content/drive/MyDrive'

# ============================================================
# Derived paths
# ============================================================
REPO_DIR    = f'{BASE_ROOT}/{PROJECT_SLUG}'
DRIVE_ROOT  = f'{BASE_ROOT}/{PROJECT_SLUG}_colab'
VENV_DIR    = f'/content/venvs/{PROJECT_SLUG}'
CONFIG_FILE = f'{REPO_DIR}/colab/config.env'
BOOTSTRAP   = f'{REPO_DIR}/colab/bootstrap/scripts/bootstrap_env.sh'

# ============================================================
# Auto-bootstrap nếu runtime mới (venv chưa có)
# ============================================================
if not os.path.isfile(f'{VENV_DIR}/bin/python'):
    print('[INFO] Venv chưa có. Chạy bootstrap (restore từ snapshot nếu có)...')
    if not os.path.isfile(BOOTSTRAP):
        raise FileNotFoundError(
            f'Không tìm thấy {BOOTSTRAP}. Chạy:\n'
            f'  !git -C {REPO_DIR} submodule update --init --recursive'
        )
    subprocess.run(['bash', BOOTSTRAP], check=True)

assert os.path.isfile(f'{VENV_DIR}/bin/python'), 'Bootstrap failed — venv không có sau bootstrap'

# ============================================================
# Helper: chạy command với venv Python
# ============================================================
VENV_PY = f'{VENV_DIR}/bin/python'

def venv_run(args, **kwargs):
    """Wrapper subprocess.run với venv python.

    Ví dụ:
        venv_run(['-m', 'pip', 'list'])
        venv_run(['my_script.py', '--arg', 'value'], cwd=REPO_DIR)
    """
    return subprocess.run([VENV_PY, *args], check=True, **kwargs)

print(f'Repo dir   : {REPO_DIR}')
print(f'Drive root : {DRIVE_ROOT}')
print(f'Venv dir   : {VENV_DIR}')
print(f'Venv py    : {VENV_PY}')

---

## Task code

> **TODO**: Thay phần dưới đây bằng task chính của project. Một số pattern phổ biến:

**A. Chạy Python script trong repo**:
```python
venv_run(['scripts/train.py', '--config', f'{DRIVE_ROOT}/config.yaml'], cwd=REPO_DIR)
```

**B. Chạy module qua `python -m`**:
```python
venv_run(['-m', 'my_package.cli', 'train', '--epochs', '100'])
```

**C. Bash multi-line (ENV vars + paths)**:
```python
%%bash -s "$VENV_DIR" "$REPO_DIR" "$DRIVE_ROOT"
set -euo pipefail
VENV="$1"; REPO="$2"; DRIVE="$3"
cd "$REPO"
"$VENV/bin/python" -m my_pkg --output "$DRIVE/output"
```

**D. Long-running task + log file**:
```python
%%bash -s "$VENV_DIR" "$DRIVE_ROOT"
VENV="$1"; DRIVE="$2"
mkdir -p "$DRIVE/logs"
"$VENV/bin/python" my_long_task.py 2>&1 | tee -a "$DRIVE/logs/task.log"
```

**E. Background server + tunnel** (xem [fastapi-server-cloudflared example](https://github.com/...)):
```python
!bash colab/scripts/run_server.sh &
!bash colab/scripts/cloudflared_start.sh
```

In [ ]:
# TODO: Thay cell này bằng code task của project.
#
# Available variables (đã set ở cell trên):
#   REPO_DIR      — repo root trên Drive
#   DRIVE_ROOT    — artifacts root trên Drive
#   VENV_DIR      — venv path (/content/venvs/<slug>)
#   VENV_PY       — đường dẫn tới python trong venv
#   venv_run(args, **kwargs) — wrapper subprocess.run với venv python
#
# Ví dụ placeholder:
venv_run(['-c', 'import sys; print("Python:", sys.version)'])
venv_run(['-c', 'import torch; print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())'])